In [1]:
!pip -q install datasets pandas

In [2]:
from datasets import load_dataset
import pandas as pd

# Load FinanceBench
dataset = load_dataset("PatronusAI/financebench", split="train")
df = dataset.to_pandas()

# Inspect dataset
print("Total questions:", len(df))
print("Columns:", df.columns.tolist())

display(df.head(5))

README.md:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

financebench_merged.jsonl:   0%|          | 0.00/958k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

Total questions: 150
Columns: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']


,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_03029,3M,3M_2018_10K,metrics-generated,Information extraction,None,What is the FY2018 capital expenditure amount ...,$1577.00,The metric capital expenditures was directly e...,OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
1,financebench_id_04672,3M,3M_2018_10K,metrics-generated,Information extraction,None,Assume that you are a public equities analyst....,$8.70,"The metric ppne, net was directly extracted fr...",OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
2,financebench_id_00499,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasoning),dg06,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",CAPEX/Revenue\nFixed Assets/Total Assets\nROA=...,OPEN_SOURCE,[{'evidence_text': '3M Company and Subsidiarie...,Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...
3,financebench_id_01226,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasonin...,dg17,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,None,OPEN_SOURCE,"[{'evidence_text': 'SG&A, measured as a percen...",Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...
4,financebench_id_01865,3M,3M_2022_10K,novel-generated,None,None,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,None,OPEN_SOURCE,[{'evidence_text': 'Worldwide Sales Change By...,Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...


In [3]:
df.columns

Index(['financebench_id', 'company', 'doc_name', 'question_type',
       'question_reasoning', 'domain_question_num', 'question', 'answer',
       'justification', 'dataset_subset_label', 'evidence', 'gics_sector',
       'doc_type', 'doc_period', 'doc_link'],
      dtype='object')

In [4]:

# Dataset overview
print("Total questions:", len(df))
print("Unique companies:", df["company"].nunique())
print("Unique documents:", df["doc_name"].nunique())

print("\n--- Question types ---")
print(df["question_type"].value_counts(dropna=False))

print("\n--- Reasoning categories ---")
print(df["question_reasoning"].value_counts(dropna=False))

print("\n--- Questions by reporting year ---")
print(df["doc_period"].value_counts(dropna=False).sort_index())

print("\n--- Questions by company ---")
print(df["company"].value_counts())


Total questions: 150
Unique companies: 32
Unique documents: 84

--- Question types ---
question_type
metrics-generated    50
domain-relevant      50
novel-generated      50
Name: count, dtype: int64

--- Reasoning categories ---
question_reasoning
None                                                                                            50
Numerical reasoning                                                                             43
Information extraction                                                                          31
Numerical reasoning OR Logical reasoning                                                         6
Logical reasoning (based on numerical reasoning)                                                 5
Logical reasoning (based on numerical reasoning) OR Logical reasoning                            5
Logical reasoning (based on numerical reasoning) OR Numerical reasoning OR Logical reasoning     4
Numerical reasoning OR information extraction              

In [5]:

print("Total questions:", len(df))
print("Unique question IDs:", df["financebench_id"].nunique())
print("Duplicate IDs:", df["financebench_id"].duplicated().sum())

print("\nMissing values:")
print(df[[
    "financebench_id",
    "question",
    "answer",
    "company",
    "doc_name",
    "doc_period",
    "evidence"
]].isna().sum())

print("\nUnique source documents:", df["doc_name"].nunique())


Total questions: 150
Unique question IDs: 150
Duplicate IDs: 0

Missing values:
financebench_id    0
question           0
answer             0
company            0
doc_name           0
doc_period         0
evidence           0
dtype: int64

Unique source documents: 84


In [6]:

# Stratified sampling by original FinanceBench question type

import pandas as pd

SEED = 42

# Target allocation: 27 + 27 + 26 = 80
allocation = {
    "metrics-generated": 27,
    "domain-relevant": 27,
    "novel-generated": 26
}

selected_parts = []

for category, count in allocation.items():
    subset = df[df["question_type"] == category]

    sampled = subset.sample(
        n=count,
        random_state=SEED
    )

    selected_parts.append(sampled)

eval_df = pd.concat(selected_parts, ignore_index=True)

# Shuffle final dataset reproducibly
eval_df = eval_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print("Selected questions:", len(eval_df))
print("\nQuestion-type distribution:")
print(eval_df["question_type"].value_counts())

print("\nUnique companies:", eval_df["company"].nunique())
print("Unique reporting years:", eval_df["doc_period"].nunique())
print("Unique source documents:", eval_df["doc_name"].nunique())

display(eval_df[[
    "financebench_id",
    "company",
    "doc_name",
    "question_type",
    "question"
]].head(10))


Selected questions: 80

Question-type distribution:
question_type
domain-relevant      27
metrics-generated    27
novel-generated      26
Name: count, dtype: int64

Unique companies: 31
Unique reporting years: 10
Unique source documents: 58


,financebench_id,company,doc_name,question_type,question
0,financebench_id_00746,Ulta Beauty,ULTABEAUTY_2023_10K,domain-relevant,Which debt securities are registered to trade ...
1,financebench_id_05718,American Water Works,AMERICANWATERWORKS_2020_10K,metrics-generated,How much (in USD billions) did American Water ...
2,financebench_id_10130,Corning,CORNING_2020_10K,metrics-generated,Based on the information provided primarily in...
3,financebench_id_01028,American Express,AMERICANEXPRESS_2022_10K,domain-relevant,What are the geographies that American Express...
4,financebench_id_06247,Walmart,WALMART_2018_10K,metrics-generated,What is FY2018 days payable outstanding (DPO) ...
5,financebench_id_00552,Microsoft,MICROSOFT_2023_10K,domain-relevant,Has Microsoft increased its debt on balance sh...
6,financebench_id_03069,AMD,AMD_2015_10K,metrics-generated,Answer the following question as if you are an...
7,financebench_id_00283,Pfizer,Pfizer_2023Q2_10Q,novel-generated,How much does Pfizer expect to pay to spin off...
8,financebench_id_04660,Block,BLOCK_2016_10K,metrics-generated,"Considering the data in the balance sheet, wha..."
9,financebench_id_03849,MGM Resorts,MGMRESORTS_2020_10K,metrics-generated,What is the FY2018 - FY2020 3 year average of ...


In [7]:

# Inspect reasoning types in our selected 80 questions

print("Reasoning categories in selected questions:")
print(eval_df["question_reasoning"].value_counts(dropna=False))

# Preliminary classification based on FinanceBench reasoning labels
def classify_question(reasoning):
    reasoning = str(reasoning).lower()

    if "numerical reasoning" in reasoning:
        return "numerical"

    if "logical reasoning" in reasoning:
        return "numerical_or_logical_review"

    if "information extraction" in reasoning:
        return "lookup"

    return "needs_manual_review"

eval_df["question_category"] = eval_df[
    "question_reasoning"
].apply(classify_question)

print("\nPreliminary classification:")
print(eval_df["question_category"].value_counts())

# Display questions requiring manual review
display(
    eval_df.loc[
        eval_df["question_category"].isin(
            ["needs_manual_review", "numerical_or_logical_review"]
        ),
        [
            "financebench_id",
            "question",
            "question_reasoning",
            "question_category"
        ]
    ]
)


Reasoning categories in selected questions:
question_reasoning
None                                                                                            26
Numerical reasoning                                                                             24
Information extraction                                                                          17
Logical reasoning (based on numerical reasoning)                                                 5
Numerical reasoning OR Logical reasoning                                                         3
Logical reasoning (based on numerical reasoning) OR Logical reasoning                            2
Information extraction OR Logical reasoning OR                                                   1
Numerical reasoning OR information extraction                                                    1
Logical reasoning (based on numerical reasoning) OR Numerical reasoning OR Logical reasoning     1
Name: count, dtype: int64

Preliminary classif

,financebench_id,question,question_reasoning,question_category
7,financebench_id_00283,How much does Pfizer expect to pay to spin off...,None,needs_manual_review
12,financebench_id_00757,Did AMD report customer concentration in FY22?,None,needs_manual_review
14,financebench_id_01935,What was the key agenda of the AMCOR's 8k fili...,None,needs_manual_review
16,financebench_id_00494,What production rate changes is Boeing forecas...,None,needs_manual_review
17,financebench_id_00299,Which of JPM's business segments had the lowes...,None,needs_manual_review
18,financebench_id_00702,Were there any potential events that are not i...,None,needs_manual_review
21,financebench_id_00460,Was there any change in the number of Best Buy...,None,needs_manual_review
23,financebench_id_01148,What industry does AMCOR primarily operate in?,Information extraction OR Logical reasoning OR,numerical_or_logical_review
25,financebench_id_00407,Which type of debt received the largest invest...,None,needs_manual_review
27,financebench_id_01902,Which Best Buy product category performed the ...,None,needs_manual_review


In [8]:

# Show questions requiring manual classification

review_df = eval_df[
    eval_df["question_category"].isin([
        "needs_manual_review",
        "numerical_or_logical_review"
    ])
].copy()

pd.set_option("display.max_colwidth", None)

display(
    review_df[[
        "financebench_id",
        "question",
        "answer",
        "question_reasoning"
    ]]
)


,financebench_id,question,answer,question_reasoning
7,financebench_id_00283,How much does Pfizer expect to pay to spin off Upjohn in the future in USD million?,77.78,None
12,financebench_id_00757,Did AMD report customer concentration in FY22?,"Yes, one customer accounted for 16% of consolidated net revenue",None
14,financebench_id_01935,What was the key agenda of the AMCOR's 8k filing dated 1st July 2022?,"Amcor Finance (USA), Inc. and Amcor Flexibles North America, Inc., entered into supplemental indentures relating to Guaranteed Senior Notes due 2026 and 2028. This involved the substitution of the Substitute Issuer (Amcor Flexibles North America) for the Former Issuer (Amcor Finance) and the assumption of covenants under the indentures. (In essence a novation agreement)",None
16,financebench_id_00494,What production rate changes is Boeing forecasting for FY2023?,"Boeing forecasts an increase in the production rates for the 737, 777X and 787 aircrafts in 2023.",None
17,financebench_id_00299,Which of JPM's business segments had the lowest net revenue in 2021 Q1?,Corporate. Its net revenue was -$473 million.,None
18,financebench_id_00702,Were there any potential events that are not in Pfizer's standard business operations that substantially increased net income in 2019?,"Yes, the gain on completion of Consumer Healthcare JV Transaction",None
21,financebench_id_00460,Was there any change in the number of Best Buy stores between Q2 of FY2024 and FY2023?,"Yes, there is decline in number stores by 1.32% from 982 stores in Q2 FY 2023 to 969 by the end of Q2 FY2024.",None
23,financebench_id_01148,What industry does AMCOR primarily operate in?,Amcor is a global leader in packaging production for various use cases.,Information extraction OR Logical reasoning OR
25,financebench_id_00407,Which type of debt received the largest investment among the short term investments for MGM in H1 FY2023?,the biggest short term investment is in corporate bonds (almost 82% of the total investment),None
27,financebench_id_01902,Which Best Buy product category performed the best (by top line) in the domestic (USA) Market during Q2 of FY2024?,"The entertainment segment experienced the highest growth of 9% during Q2 FY2024, primarily from gaming division.",None


In [9]:
display(
    review_df[[
        "financebench_id",
        "question",
        "answer",
        "question_reasoning"
    ]].iloc[21:]
)

,financebench_id,question,answer,question_reasoning
64,financebench_id_00859,"Among all of the derivative instruments that Verizon used to manage the exposure to fluctuations of foreign currencies exchange rates or interest rates, which one had the highest notional value in FY 2021?","Cross currency swaps. Its notional value was $32,502 million.",None
65,financebench_id_00651,Is growth in JnJ's adjusted EPS expected to accelerate in FY2023?,"No, rate of growth in adjusted EPS is expected to decelerate slightly from 3.6% in FY2022 to 3.5% in FY2023.",None
66,financebench_id_01930,"How much was the Real change in Sales for AMCOR in FY 2023 vs FY 2022, if we exclude the impact of FX movement, passthrough costs and one-off items?",The Real Growth was flat in FY 2023 vs FY 2022.,None
74,financebench_id_01964,What was the largest liability in American Express's Balance Sheet in 2022?,Customer deposits,None
76,financebench_id_02119,"If JPM went bankrupted by the end by 2021 Q1 and liquidated all of its assets to pay its shareholders, how much could each shareholder get?",They could receive $66.56 per share.,None
77,financebench_id_00605,What percent of Ulta Beauty's total spend on stock repurchases for FY 2023 occurred in Q4 of FY2023?,"36%. The answer here assumes FY2023 refers to the 12 months ended on January 28, 2023 (although the company refers to this period as its fiscal 2022.",None


In [10]:

# Manual classifications for the 27 reviewed questions
manual_labels = {
    "00283": "lookup",
    "00757": "lookup",
    "01935": "lookup",
    "00494": "lookup",
    "00299": "lookup",
    "00702": "lookup",
    "00460": "numerical",
    "01148": "lookup",
    "00407": "lookup",
    "01902": "lookup",
    "01911": "numerical",
    "01865": "lookup",
    "01476": "numerical",
    "01487": "numerical",
    "01936": "lookup",
    "00606": "numerical",
    "00839": "lookup",
    "01912": "lookup",
    "00705": "numerical",
    "00603": "lookup",
    "00585": "lookup",
    "00859": "lookup",
    "00651": "lookup",
    "01930": "needs_evidence_review",
    "01964": "lookup",
    "02119": "numerical",
    "00605": "numerical"
}

# Extract the numerical portion of the FinanceBench ID
eval_df["short_id"] = (
    eval_df["financebench_id"]
    .str.extract(r"(\d+)$")[0]
)

# Apply manual classifications
eval_df["final_category"] = eval_df.apply(
    lambda row: manual_labels.get(
        row["short_id"],
        row["question_category"]
    ),
    axis=1
)

print("Classification results:")
print(eval_df["final_category"].value_counts())

print("\nQuestions needing evidence review:")
display(eval_df.loc[
    eval_df["final_category"] == "needs_evidence_review",
    ["financebench_id", "question", "answer"]
])


Classification results:
final_category
numerical                44
lookup                   35
needs_evidence_review     1
Name: count, dtype: int64

Questions needing evidence review:


,financebench_id,question,answer
66,financebench_id_01930,"How much was the Real change in Sales for AMCOR in FY 2023 vs FY 2022, if we exclude the impact of FX movement, passthrough costs and one-off items?",The Real Growth was flat in FY 2023 vs FY 2022.


In [11]:

# Assign a provisional category to the remaining question
eval_df.loc[
    eval_df["financebench_id"] == "financebench_id_01930",
    "final_category"
] = "numerical"

# Check final distribution
print("Updated question distribution:")
print(eval_df["final_category"].value_counts())

# Validate classification
assert len(eval_df) == 80
assert eval_df["final_category"].isin(
    ["lookup", "numerical"]
).all()

print("\nAll 80 questions have a category.")


Updated question distribution:
final_category
numerical    45
lookup       35
Name: count, dtype: int64

All 80 questions have a category.


In [12]:

# Check source documents required by our 80 questions

required_docs = eval_df[
    ["company", "doc_name", "doc_period", "doc_link"]
].drop_duplicates(subset=["doc_name"]).copy()

print("Total unique documents required:", len(required_docs))

print("\nDocuments with missing PDF/source links:")
print(required_docs["doc_link"].isna().sum())

print("\nDocuments with empty links:")
print(
    required_docs["doc_link"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("\nSample of required documents:")
display(required_docs.head(10))


Total unique documents required: 58

Documents with missing PDF/source links:
0

Documents with empty links:
0

Sample of required documents:


,company,doc_name,doc_period,doc_link
0,Ulta Beauty,ULTABEAUTY_2023_10K,2023,https://www.ulta.com/investor/sec-filings/annual-reports/content/0001558370-23-004581/0001558370-23-004581.pdf
1,American Water Works,AMERICANWATERWORKS_2020_10K,2020,https://d18rn0p25nwr6d.cloudfront.net/CIK-0001410636/088387fd-7250-4370-80a7-dc5a30ea84ed.pdf
2,Corning,CORNING_2020_10K,2020,https://d18rn0p25nwr6d.cloudfront.net/CIK-0000024741/7cdeac7f-277e-4258-ad8e-4ba0f655f368.pdf
3,American Express,AMERICANEXPRESS_2022_10K,2022,https://s26.q4cdn.com/747928648/files/doc_financials/2022/q4/AMEX-10K-4Q22.pdf
4,Walmart,WALMART_2018_10K,2018,https://d18rn0p25nwr6d.cloudfront.net/CIK-0000104169/a25e7acb-aa07-49f3-8c0c-0c69e5a8d372.pdf
5,Microsoft,MICROSOFT_2023_10K,2023,https://microsoft.gcs-web.com/static-files/e2931fdb-9823-4130-b2a8-f6b8db0b15a9
6,AMD,AMD_2015_10K,2015,https://ir.amd.com/sec-filings/filter/annual-filings/content/0000002488-16-000111/0000002488-16-000111.pdf
7,Pfizer,Pfizer_2023Q2_10Q,2023,https://d18rn0p25nwr6d.cloudfront.net/CIK-0000078003/c7518e4a-3aaf-4987-ae85-27460d0f526c.pdf
8,Block,BLOCK_2016_10K,2016,https://d18rn0p25nwr6d.cloudfront.net/CIK-0001512673/1862b88f-4e6b-4251-be45-4833fc0a0126.pdf
9,MGM Resorts,MGMRESORTS_2020_10K,2020,https://d18rn0p25nwr6d.cloudfront.net/CIK-0000789570/03692647-6ccc-4177-8ce9-13ac35692254.pdf


In [13]:

import requests
import time

results = []

for _, row in required_docs.iterrows():
    url = row["doc_link"]

    try:
        response = requests.get(
            url,
            headers={"User-Agent": "FinanceBenchResearch/1.0"},
            timeout=20,
            stream=True,
            allow_redirects=True
        )

        status = response.status_code
        content_type = response.headers.get(
            "Content-Type", ""
        ).lower()

        # Check file signature rather than trusting URL extension
        first_bytes = next(
            response.iter_content(chunk_size=1024),
            b""
        )
        is_pdf = first_bytes.lstrip().startswith(b"%PDF-")

        results.append({
            "doc_name": row["doc_name"],
            "status_code": status,
            "is_pdf": is_pdf,
            "content_type": content_type
        })

        response.close()

    except requests.RequestException as e:
        results.append({
            "doc_name": row["doc_name"],
            "status_code": None,
            "is_pdf": False,
            "content_type": str(e)[:100]
        })

    time.sleep(0.2)

pdf_check_df = pd.DataFrame(results)

print("Total documents:", len(pdf_check_df))
print("Accessible PDFs:", pdf_check_df["is_pdf"].sum())

print("\nStatus codes:")
print(pdf_check_df["status_code"].value_counts(dropna=False))

print("\nDocuments requiring attention:")
display(pdf_check_df[~pdf_check_df["is_pdf"]])


Total documents: 58
Accessible PDFs: 42

Status codes:
status_code
200.0    45
403.0    10
NaN       3
Name: count, dtype: int64

Documents requiring attention:


,doc_name,status_code,is_pdf,content_type
5,MICROSOFT_2023_10K,403.0,False,text/html
6,AMD_2015_10K,200.0,False,text/html; charset=utf-8
29,JOHNSON_JOHNSON_2022_10K,NaN,False,"HTTPSConnectionPool(host='johnsonandjohnson.gcs-web.com', port=443): Max retries exceeded with url:"
30,PEPSICO_2022_10K,403.0,False,text/html
32,PEPSICO_2023Q1_EARNINGS,200.0,False,text/html
35,JOHNSON_JOHNSON_2023Q2_EARNINGS,NaN,False,"HTTPSConnectionPool(host='johnsonandjohnson.gcs-web.com', port=443): Max retries exceeded with url:"
36,ADOBE_2022_10K,403.0,False,text/html
37,KRAFTHEINZ_2019_10K,200.0,False,text/html
41,FOOTLOCKER_2022_8K_dated_2022-08-19,403.0,False,text/html
42,ADOBE_2017_10K,403.0,False,text/html


In [14]:

import requests

# Retrieve the official FinanceBench PDF filenames
api_url = "https://api.github.com/repos/patronus-ai/financebench/contents/pdfs"

response = requests.get(api_url, timeout=30)
response.raise_for_status()

github_pdfs = response.json()

pdf_names = {
    item["name"] for item in github_pdfs
    if item["name"].lower().endswith(".pdf")
}

print("PDFs available on GitHub:", len(pdf_names))
print("\nSample filenames:")
print(sorted(pdf_names)[:15])

print("\nDocuments requiring attention:",
      len(pdf_check_df[~pdf_check_df["is_pdf"]]))


PDFs available on GitHub: 368

Sample filenames:
['3M_2015_10K.pdf', '3M_2016_10K.pdf', '3M_2017_10K.pdf', '3M_2018_10K.pdf', '3M_2019_10K.pdf', '3M_2020_10K.pdf', '3M_2021_10K.pdf', '3M_2022_10K.pdf', '3M_2023Q2_10Q.pdf', 'ACTIVISIONBLIZZARD_2015_10K.pdf', 'ACTIVISIONBLIZZARD_2016_10K.pdf', 'ACTIVISIONBLIZZARD_2017_10K.pdf', 'ACTIVISIONBLIZZARD_2018_10K.pdf', 'ACTIVISIONBLIZZARD_2019_10K.pdf', 'ACTIVISIONBLIZZARD_2020_10K.pdf']

Documents requiring attention: 16


In [15]:

# Match required documents against official FinanceBench PDFs

required_docs["expected_pdf"] = (
    required_docs["doc_name"].astype(str) + ".pdf"
)

required_docs["available_on_github"] = (
    required_docs["expected_pdf"].isin(pdf_names)
)

print("Total required documents:", len(required_docs))

print(
    "Available on FinanceBench GitHub:",
    required_docs["available_on_github"].sum()
)

print(
    "Missing from FinanceBench GitHub:",
    (~required_docs["available_on_github"]).sum()
)

print("\nDocuments not matched:")
display(
    required_docs.loc[
        ~required_docs["available_on_github"],
        ["doc_name", "expected_pdf"]
    ]
)


Total required documents: 58
Available on FinanceBench GitHub: 58
Missing from FinanceBench GitHub: 0

Documents not matched:


,doc_name,expected_pdf


In [17]:


import json
from pathlib import Path

# Start with our selected 80 questions
export_df = eval_df.copy()

# Remove the old preliminary classification
export_df = export_df.drop(
    columns=["question_category"],
    errors="ignore"
)

# Rename the finalized classification and question ID
export_df = export_df.rename(columns={
    "financebench_id": "question_id",
    "final_category": "question_category"
})

# Select the fields required for evaluation
columns = [
    "question_id",
    "question",
    "company",
    "doc_name",
    "doc_period",
    "question_type",
    "question_category",
    "question_reasoning",
    "answer",
    "evidence"
]

export_df = export_df[columns]

# Validate the dataset
assert len(export_df) == 80
assert export_df.columns.is_unique
assert export_df["question_id"].is_unique
assert export_df["question_category"].isin(
    ["lookup", "numerical"]
).all()

# Create output directory
Path("data").mkdir(exist_ok=True)

# Export JSONL (overwrite previous incorrect file)
output_path = Path("data/eval_set.jsonl")

with open(output_path, "w", encoding="utf-8") as f:
    for record in export_df.to_dict(orient="records"):
        f.write(json.dumps(record, default=str) + "\n")

print("Saved successfully:", output_path)
print("Total questions:", len(export_df))

print("\nQuestion categories:")
print(export_df["question_category"].value_counts())

print("\nFirst record:")
print(
    json.dumps(
        export_df.iloc[0].to_dict(),
        indent=2,
        default=str
    )[:2500]
)



Saved successfully: data/eval_set.jsonl
Total questions: 80

Question categories:
question_category
numerical    45
lookup       35
Name: count, dtype: int64

First record:
{
  "question_id": "financebench_id_00746",
  "question": "Which debt securities are registered to trade on a national securities exchange under Ulta Beauty's name as of FY2023?",
  "company": "Ulta Beauty",
  "doc_name": "ULTABEAUTY_2023_10K",
  "doc_period": 2023,
  "question_type": "domain-relevant",
  "question_category": "lookup",
  "question_reasoning": "Information extraction",
  "answer": "There are none",
  "evidence": "[{'evidence_text': 'UNITED STATES\\nSECURITIES AND EXCHANGE COMMISSION\\nWashington, DC 20549\\nFORM 10-K\\n Annual Report Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934\\nFor the fiscal year ended January 28, 2023\\nor\\n Transition Report Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934\\nFor the transition period from _____________ to ________

In [19]:

# Check the different evidence data types
print("Evidence data types:")
print(export_df["evidence"].apply(lambda x: type(x).__name__).value_counts())

# Display a sample of each type
for type_name in export_df["evidence"].apply(lambda x: type(x).__name__).unique():
    sample = export_df.loc[
        export_df["evidence"].apply(lambda x: type(x).__name__) == type_name,
        "evidence"
    ].iloc[0]

    print(f"\nType: {type_name}")
    print("Sample:", repr(sample)[:800])


Evidence data types:
evidence
ndarray    80
Name: count, dtype: int64

Type: ndarray
Sample: array([{'evidence_text': 'UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, DC 20549\nFORM 10-K\n Annual Report Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934\nFor the fiscal year ended January 28, 2023\nor\n Transition Report Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934\nFor the transition period from _____________ to _____________\nCommission File Number: 001-33764\nULTA BEAUTY, INC.\n(Exact name of registrant as specified in its charter)\nDelaware\n(State or other jurisdiction of\nincorporation or organization)\n38-4022268\n(I.R.S. Employer\nIdentification No.)\n1000 Remington Blvd., Suite 120\nBolingbrook, Illinois\n(Address of principal executive offices)\n60440\n(Zip code)\nRegistrants telephone number, including area cod


In [20]:

import numpy as np
import ast
import json
from pathlib import Path

def format_evidence(value):
    # Convert NumPy arrays into Python lists
    if isinstance(value, np.ndarray):
        value = value.tolist()

    # Handle evidence stored as text
    if isinstance(value, str):
        value = ast.literal_eval(value)

    # Handle a single evidence dictionary
    if isinstance(value, dict):
        value = [value]

    if not isinstance(value, (list, tuple)):
        raise ValueError(
            f"Unexpected evidence type: {type(value)}"
        )

    formatted = []

    for item in value:
        formatted.append({
            "doc_name": item["doc_name"],
            "page_number": int(item["evidence_page_num"])
        })

    return formatted

# Convert the evidence entries
export_df["evidence"] = export_df["evidence"].apply(
    format_evidence
)

# Validate the converted evidence
assert export_df["evidence"].apply(len).gt(0).all()

# Save the corrected JSONL file
output_path = Path("data/eval_set.jsonl")
output_path.parent.mkdir(exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for record in export_df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved successfully:", output_path)
print("Total questions:", len(export_df))

print("\nFirst record:")
print(json.dumps(
    export_df.iloc[0].to_dict(),
    indent=2,
    ensure_ascii=False
))


Saved successfully: data/eval_set.jsonl
Total questions: 80

First record:
{
  "question_id": "financebench_id_00746",
  "question": "Which debt securities are registered to trade on a national securities exchange under Ulta Beauty's name as of FY2023?",
  "company": "Ulta Beauty",
  "doc_name": "ULTABEAUTY_2023_10K",
  "doc_period": 2023,
  "question_type": "domain-relevant",
  "question_category": "lookup",
  "question_reasoning": "Information extraction",
  "answer": "There are none",
  "evidence": [
    {
      "doc_name": "ULTABEAUTY_2023_10K",
      "page_number": 0
    }
  ]
}


In [21]:
import json
from pathlib import Path

file_path = Path("data/eval_set.jsonl")

with open(file_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

required_fields = {
    "question_id", "question", "company",
    "doc_name", "doc_period", "question_type",
    "question_category", "answer", "evidence"
}

assert len(records) == 80
assert len({r["question_id"] for r in records}) == 80
assert all(required_fields.issubset(r.keys()) for r in records)
assert all(r["question_category"] in ("lookup", "numerical") for r in records)
assert all(
    isinstance(r["evidence"], list)
    and len(r["evidence"]) > 0
    and all(
        "doc_name" in e and isinstance(e.get("page_number"), int)
        for e in r["evidence"]
    )
    for r in records
)

print("All validation checks passed!")
print("Total valid records:", len(records))
print("Unique question IDs:", len({r["question_id"] for r in records}))
print("File size:", round(file_path.stat().st_size / 1024, 2), "KB")

All validation checks passed!
Total valid records: 80
Unique question IDs: 80
File size: 46.61 KB


In [22]:
from google.colab import files

files.download("data/eval_set.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>